In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_events = spark.table("workspace.bronze.events")
bronze_venues = spark.table("workspace.bronze.venues")
bronze_orgs   = spark.table("workspace.bronze.organizations")

silver_events = (
    bronze_events.alias("e")
    .join(bronze_venues.alias("v"), F.col("v.event_id") == F.col("e.id"), "left")
    .join(bronze_orgs.alias("o"),   F.col("o.id") == F.col("e.org_id"),   "left")
    .select(
        F.col("e.id").alias("event_id"),
        F.col("e.title"),
        F.col("e.status"),
        F.col("e.visibility"),
        F.col("e.is_online"),
        F.col("e.max_attendees"),
        F.col("e.start_date"),
        F.col("e.end_date"),
        F.col("e.timezone"),
        # Duração derivada
        (F.unix_timestamp("e.end_date") - F.unix_timestamp("e.start_date"))
            .alias("duration_seconds"),
        # Venue
        F.col("v.name").alias("venue_name"),
        F.col("v.city").alias("venue_city"),
        F.col("v.state").alias("venue_state"),
        F.col("v.capacity").alias("venue_capacity"),
        F.col("v.lat"),
        F.col("v.lng"),
        # Org
        F.col("e.org_id"),
        F.col("o.name").alias("org_name"),
        # Metadata
        F.col("e.created_at"),
        F.col("e._ingested_at"),
    )
    .dropDuplicates(["event_id"])
)

(silver_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.events"))